# E004-E007: error-driven blocking (blocking only, no model CV)

| step | what | output |
|---|---|---|
| A | **Probe**: 10k train S1 per country vs the FULL country pool, every channel; E003 misses bucketed by failure mode, with which channel recovers them | `probe.json`, `pairs_*.parquet`, `misses_*.parquet` |
| B | **Select E003-E007**: pooled greedy union (no RRF, no cut; recall measured on the union) | `experiments.json`, `e007_spec.txt` |
| C | **Full-density baseline**: E003 blocker on ALL 2.2M train S1: recall, recall@k, oracle, present/absent split, error buckets, runtime, peak RAM | `blocking_full.json`, `ERROR_BUCKETS.md` |
| D | **E007 on ALL train S1**: the chosen union measured for real (count, recall, runtime, RAM) | `blocking_full.json` |
| E | Reports + one small bundle to send back | `E004_results.tgz` |

Channels: `base` (E003), `addr_only`, `name_noaddr`, `conj` (token-pair keys), `rescue` (exact blocks),
`bm25` (loose df cap), `bge_native` (BGE-M3 on native-script pool records only, GPU), `e5_native` (ablation),
plus `base_wide:200` as a reference for "what more K buys".

**Settings (right panel):** Accelerator **GPU T4 x2** (needed only for BGE-M3/e5; with None they are skipped).
Internet **On**. Persistence **Files**. Input: the competition dataset. Optional: a saved dataset containing
`work/prepared/train/*.parquet` and/or `work/train/cand_k45_n10_df2500.parquet` from an earlier run; both are reused.

**Resume:** every step writes to `/kaggle/working` and is skipped if its output exists. If the session dies,
*Run all* again.

In [ ]:
# 1. Config
N_QUERIES   = 10000                      # probe queries per country (recall SE ~0.3 pt)
PROBE_CH    = "base,base_wide:200,addr_only:20,name_noaddr:10,conj:50,rescue,bm25:50"
DENSE_CH    = "bge_native:20,e5_native:20"   # added automatically when a GPU is present
MIN_GAIN, MIN_EFF, MAX_PAIRS = 0.002, 0.0005, 80   # greedy selection rule (see EXPERIMENT_COMPARISON.md)
E003        = dict(k_comb=45, k_name=10, df_cap=2500)
RUN_BASELINE_ALL_S1 = True               # step C
RUN_E007_ALL_S1     = True               # step D
E007_SPEC_OVERRIDE  = ""                 # e.g. "base,conj:20,bm25:20"; empty = use the greedy choice
REPO     = "https://github.com/Bexwane/AmazonMLchallenge.git"
CODE_DIR = "/kaggle/working/ber"
WORK     = "/kaggle/working/work"
PROBE    = "/kaggle/working/probe_E004"
REPORTS  = "/kaggle/working/reports_E004"

In [ ]:
# 2. Locate dataset; reuse saved caches if attached
import glob, os
hits = glob.glob("/kaggle/input/**/train/train_source1.tsv", recursive=True)
assert hits, "Dataset not found: add the dataset with train/ and test/ folders as notebook input"
DATA = os.path.dirname(os.path.dirname(hits[0]))
print("DATA =", DATA)
def reuse(pattern, dest_dir):
    found = [p for p in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)]
    for p in found:
        d = os.path.join(dest_dir, os.path.basename(p))
        if not os.path.exists(d):
            os.makedirs(dest_dir, exist_ok=True); os.symlink(p, d); print("reusing", p)
reuse("prepared/train/s1.parquet", f"{WORK}/prepared/train")
reuse("prepared/train/s23.parquet", f"{WORK}/prepared/train")
reuse("train/cand_k45_n10_df2500.parquet", f"{WORK}/train")
!free -g; nproc; nvidia-smi -L 2>/dev/null || echo "no GPU"

In [ ]:
# 3. Code, dependencies, unit tests
!rm -rf {CODE_DIR} && git clone -q {REPO} {CODE_DIR} && cd {CODE_DIR} && git log --oneline -1
!pip install -q rapidfuzz==3.14.6
import sys; sys.path.insert(0, f"{CODE_DIR}/src")
import torch
GPU = torch.cuda.is_available()
if GPU:
    !pip install -q -U sentence-transformers
CHANNELS = PROBE_CH + ("," + DENSE_CH if GPU else "")
print("GPU:", GPU, "| probe channels:", CHANNELS)
!cd {CODE_DIR} && python -m pytest -q tests

In [ ]:
# helper: run a script with live, timestamped output; fails loudly (exit -9 = out of memory)
import subprocess, time, json
def run(*args):
    t = time.time()
    p = subprocess.Popen(["python", *map(str, args)], cwd=CODE_DIR, env={**os.environ, "PYTHONPATH": "src"},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    assert p.wait() == 0, f"failed (exit {p.returncode}); -9 means out of memory"
    print(f"--- done in {(time.time() - t) / 60:.1f} min")

In [ ]:
# 4. Normalize train once (cached, about 30 min the first time)
run("-c", f"from ber.prepare import load_prepared; s1, s23 = load_prepared({DATA!r}, 'train', {WORK!r}); print(len(s1), len(s23))")

In [ ]:
# 5. STEP A: probe, one process per country (skipped for countries already done)
import pandas as pd
countries = sorted(pd.read_parquet(f"{WORK}/prepared/train/s1.parquet", columns=["country"])["country"].unique())
done = json.load(open(f"{PROBE}/probe.json")) if os.path.exists(f"{PROBE}/probe.json") else {}
for c in countries:
    if c in done and os.path.exists(f"{PROBE}/pairs_{c}.parquet"):
        print("probe done:", c); continue
    run("scripts/recall_probe.py", "--data", DATA, "--work", WORK, "--n", N_QUERIES, "--channels", CHANNELS,
        "--countries", c, "--out", PROBE)
    !free -g | head -2

In [ ]:
# 6. STEP B: pooled greedy selection of E003-E007 (union recall; no RRF, no truncation)
run("scripts/recall_probe.py", "--combine", "--out", PROBE, "--min-gain", MIN_GAIN, "--min-eff", MIN_EFF,
    "--max-pairs", MAX_PAIRS)
E007_SPEC = E007_SPEC_OVERRIDE or open(f"{PROBE}/e007_spec.txt").read().strip()
ex = json.load(open(f"{PROBE}/experiments.json"))
display(pd.DataFrame([{"experiment": e["experiment"], "spec": e["spec"], "pairs/S1": round(e["pairs_per_s1"], 1),
                       "recall": round(e["pair_recall"], 4), **{k: round(v, 4) for k, v in e["recall_by_country"].items()},
                       "incremental": round(e["incremental_recall"], 4)} for e in ex["experiments"]]))
# rough all-S1 runtime of E007 from the probe timings (pool build + queries scale with S1 count)
est = sum(ex["channels"][c][n]["sec"] * ex["weights"][c] for c in ex["channels"] for n in ex["channels"][c]
          if any(n == s.split(":")[0] for s in E007_SPEC.split(",")))
print("E007 spec:", E007_SPEC, f"| rough upper-bound blocking time on all S1: {est / 3600:.1f} h")

In [ ]:
# 7. STEP C: full-density E003 baseline on ALL train S1 (+ error buckets). Reuses the cached candidates if present.
BASE_JSON = f"{WORK}/experiments/E003-fullblock/blocking_full.json"
if RUN_BASELINE_ALL_S1 and not os.path.exists(BASE_JSON):
    run("scripts/eval_blocking_full.py", "--data", DATA, "--work", WORK, "--exp", "E003-fullblock",
        "--k-comb", E003["k_comb"], "--k-name", E003["k_name"], "--df-cap", E003["df_cap"], "--reports", REPORTS)
if os.path.exists(BASE_JSON):
    b = json.load(open(BASE_JSON))
    print({k: b[k] for k in ("pairs", "pairs_per_s1", "pair_recall", "recall_India", "recall_US", "oracle_macro_f05",
                             "absent", "absent_no_key", "block_min", "total_min", "peak_gib")})
    display(pd.DataFrame(b["presence"]))

In [ ]:
# 8. STEP D: the chosen E007 union on ALL train S1, measured (not estimated)
FINAL_JSON = f"{WORK}/experiments/E007-fullblock/blocking_full.json"
if RUN_E007_ALL_S1 and E007_SPEC != "base" and not os.path.exists(FINAL_JSON):
    run("scripts/eval_blocking_full.py", "--data", DATA, "--work", WORK, "--exp", "E007-fullblock",
        "--channels", E007_SPEC, "--df-cap", E003["df_cap"])
if os.path.exists(FINAL_JSON):
    f = json.load(open(FINAL_JSON))
    print({k: f[k] for k in ("blocker", "pairs", "pairs_per_s1", "pair_recall", "recall_India", "recall_US",
                             "oracle_macro_f05", "block_min", "peak_gib")})

In [ ]:
# 9. STEP E: reports + one small bundle. Download /kaggle/working/E004_results.tgz and send it back.
import shutil, tarfile
run("scripts/make_reports.py", "--baseline", BASE_JSON, "--probe", f"{PROBE}/experiments.json",
    "--final", FINAL_JSON, "--out", REPORTS)
B = "/kaggle/working/E004_results"
os.makedirs(B, exist_ok=True)
files = glob.glob(f"{REPORTS}/*.md") + glob.glob(f"{PROBE}/*.json") + glob.glob(f"{PROBE}/misses_*.parquet")
files += [(BASE_JSON, "E003_blocking_full.json"), (FINAL_JSON, "E007_blocking_full.json")]
for f in files:
    src, name = f if isinstance(f, tuple) else (f, os.path.basename(f))
    if os.path.exists(src):
        shutil.copy(src, f"{B}/{name}")
with tarfile.open("/kaggle/working/E004_results.tgz", "w:gz") as t:
    t.add(B, arcname="E004_results")
print(sorted(os.listdir(B)), os.path.getsize("/kaggle/working/E004_results.tgz") // 1024, "KiB")
print(open(f"{REPORTS}/EXPERIMENT_COMPARISON.md").read()[:4000])